# Rate arithmetic, tables, and ageing

A flow rate is an expression tree. `tanh`, `abs`, `clip` and `**` are nodes
in that tree, so a triangular seed or a detection scale-up does not have to
hide inside a `Transform` callback. The same operators apply to a saved
`Output`. A parameter-only piece such as `tanh(Param("se"))` is evaluated
with `eval_closed` and then scales the output. Per-age death rates come from
one table. A yearly mixing matrix is a `Lookup`. Ageing through uneven bands
is `TraitChain.from_breakpoints`.

The triangle should peak near t = 20 and be zero outside a width of 8.
The tanh scale-up should rise from `start` toward `end`. Scaling a
output by `tanh(se)` should lower every bar by the same factor. Death
rates should be one line per age. In 1851 both ages should share a
force of infection; in 1840, clamped to the assortative matrix, they
should not. Ageing from breakpoints should match a hand-written chain and
move people from the youngest band into older ones.


## Triangular seed

tb_macro's seeding pulse is `clip(h * (1 - abs(t - peak) / width), 0)`:
a triangle of height `h` and width `width`, zero outside that window.
The line should be that triangle, zero at both ends of the 40-day grid.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import numpy as np

from summer4 import (
    FlowModel,
    Param,
    Property,
    PropertyMap,
    Time,
    TransitionFlow,
    clip,
)

state = Property("state", ("S", "I"))
pmap = PropertyMap.from_property(state)
seed = clip(Param("height") * (1 - abs(Time() - Param("peak")) / Param("width")), 0)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("seed", state["S"], state["I"], seed, absolute=True))
compiled = model.compile()
params = {"height": 0.05, "peak": 20.0, "width": 8.0}
times = np.linspace(0.0, 40.0, 50)
y = np.array([1.0, 0.0])
got = np.array(
    [float(np.asarray(compiled.vector_field(float(t), y, params))[1]) for t in times]
)
expect = np.maximum(
    params["height"] * (1.0 - np.abs(times - params["peak"]) / params["width"]),
    0.0,
)
np.testing.assert_allclose(got, expect, rtol=1e-5, atol=1e-6)
assert got[0] == 0.0 and got[-1] == 0.0
assert got.max() > 0.9 * params["height"]
print(f"peak sample {got.max():.4f} (height {params['height']})")

pd.DataFrame({"seed rate": got}, index=times).plot(
    title="Triangular seed: height 0.05, peak at t = 20, width 8",
    labels={"index": "time", "value": "people / time"},
)


## Tanh scale-up

Detection rises from `start` to `end`, with the midpoint at `inflection` and
steepness `shape`:
`start + (end - start) * (tanh(shape * (t - inflection)) + 1) / 2`.
The curve should rise through the inflection at t = 18.


In [ ]:
from summer4 import tanh

scale = Param("start") + (Param("end") - Param("start")) * (
    tanh(Param("shape") * (Time() - Param("inflection"))) + 1
) / 2
detect = FlowModel(pmap)
detect.add_flow(
    TransitionFlow("detect", state["S"], state["I"], scale, absolute=True)
)
compiled = detect.compile()
params = {"shape": 0.3, "inflection": 18.0, "start": 0.1, "end": 1.0}
got = np.array(
    [float(np.asarray(compiled.vector_field(float(t), y, params))[1]) for t in times]
)
expect = params["start"] + (params["end"] - params["start"]) * (
    np.tanh(params["shape"] * (times - params["inflection"])) + 1.0
) / 2.0
np.testing.assert_allclose(got, expect, rtol=1e-5, atol=1e-6)
assert got[0] < got[-1]
print(f"scale-up {got[0]:.3f} -> {got[-1]:.3f}")

pd.DataFrame({"detection scale": got}, index=times).plot(
    title="Tanh scale-up from 0.1 toward 1",
    labels={"index": "time", "value": "scale"},
)


## The same transform on an output

`tanh(Param("se"))` is a rate expression. `eval_closed` evaluates the
parameter-only part against a parameter dict. The array scales an `Output`
with the same operator the rate tree would use. The output does not carry
parameters, so multiplying by the unevaluated expression is an error. The bars are
a four-point series before and after scaling by `tanh(0.4)`.


In [ ]:
from summer4 import Output, eval_closed
from summer4.time import TimeAxis

expr = tanh(Param("se"))
factor = float(eval_closed(expr, {"se": 0.4}))
np.testing.assert_allclose(factor, np.tanh(0.4), rtol=1e-5, atol=1e-6)

constant = FlowModel(pmap)
constant.add_flow(
    TransitionFlow("detect", state["S"], state["I"], expr, absolute=True)
)
rate = float(np.asarray(constant.compile().vector_field(0.0, y, {"se": 0.4}))[1])
np.testing.assert_allclose(rate, factor, rtol=1e-5, atol=1e-6)

values = np.array([10.0, 20.0, 30.0, 40.0])
output = Output(
    times=TimeAxis(values=np.linspace(0.0, 3.0, 4), epoch=None, kind="explicit"),
    values=values,
    dims=("time",),
)
scaled = output * factor
np.testing.assert_allclose(np.asarray(scaled.values), values * factor, rtol=1e-5)
try:
    output * expr
except TypeError as exc:
    assert "eval_closed" in str(exc)
else:
    raise AssertionError("output * unevaluated expr should fail")
print(f"sensitivity {factor:.4f}, scaled output {np.asarray(scaled.values)}")

pd.DataFrame(
    {"output": values, "output * tanh(0.4)": np.asarray(scaled.values)},
    index=np.linspace(0.0, 3.0, 4),
).plot.bar(
    title="eval_closed(tanh(se)) scales a saved output",
    labels={"index": "time", "value": "value"},
)


## Per-age death rates

`Interp` is one series. A death-rate table is one series per age band, and
writing each band out as its own tree is a thousand knot nodes. `Data.table`
holds the whole grid, and `interp` evaluates every column in one node. The
result is a `GroupedRate` over the age property, so an exit flow applies each
band's rate to that band. The lines are the table; the bars are the
exit rate at 2005, halfway between the 2000 and 2010 knots, with one
person in each band.


In [ ]:
import pandas as pd

from summer4 import ExitFlow, PropertyData
from summer4.data import Data

age = Property("age", ("0", "15", "65"))
alive = Property("state", ("Y",))
death_times = np.array([2000.0, 2010.0, 2020.0])
deaths = pd.DataFrame(
    {
        "0": [0.020, 0.015, 0.010],
        "15": [0.004, 0.005, 0.006],
        "65": [0.040, 0.045, 0.050],
    }
)
table = Data.table(death_times, deaths, over=age)
age_map = PropertyMap.from_property(alive).stratify(age)
deaths_model = FlowModel(age_map)
deaths_model.add_flow(ExitFlow("die", alive["Y"], table.interp()))
compiled_deaths = deaths_model.compile()
pop = PropertyData.wrap(age_map, np.ones(age_map.size))
# Halfway from 2000 to 2010, one person in each band.
mid = np.asarray(compiled_deaths.vector_field(2005.0, pop, {}).data)
expect = -0.5 * (deaths.iloc[0].to_numpy() + deaths.iloc[1].to_numpy())
np.testing.assert_allclose(mid, expect, rtol=1e-5, atol=1e-6)
after = np.asarray(compiled_deaths.vector_field(2030.0, pop, {}).data)
np.testing.assert_allclose(after, -deaths.iloc[-1].to_numpy(), rtol=1e-5, atol=1e-6)
print(f"death rates at 2005: {-mid}")

deaths.set_index(pd.Index(death_times, name="year")).plot(
    title="Death-rate table, one series per age",
    labels={"index": "year", "value": "per year"},
)
pd.Series(-mid, index=[str(trait) for trait in age.traits]).to_frame(
    "exit rate"
).plot.bar(
    title="At 2005 each band is halfway between its 2000 and 2010 rates",
    labels={"index": "age", "value": "per year"},
)


## A mixing matrix per year

The contact matrix changes by year, but it does not have to be rebuilt inside
the vector field. The stack of yearly matrices is a parameter. `Lookup`
gathers the row for `floor(Time() - 1850)`, and that matrix is the mixing
matrix. A year before the first row holds the first matrix. In 1851 the matrix
is homogeneous, so the two ages share a force of infection. In 1840
the lookup clamps to the first, assortative, matrix, and the ages
separate.


In [ ]:
from summer4 import floor
from summer4.epi import ForceOfInfection, MixingMatrix
from summer4.flows.rates import Lookup

stack = np.stack(
    [
        np.array([[0.8, 0.2], [0.3, 0.7]]),
        np.array([[0.5, 0.5], [0.5, 0.5]]),
        np.array([[0.1, 0.9], [0.4, 0.6]]),
    ]
)
age2 = Property("age", ("young", "old"))
state2 = Property("state", ("S", "I", "R"))
pmap2 = PropertyMap.from_property(state2).stratify(age2)
foi = ForceOfInfection(
    "infection",
    infectious=state2["I"],
    group_by=age2,
    kind="frequency",
    contact_rate=0.4,
    mixing=MixingMatrix(
        age2,
        Lookup(Param("stack"), floor(Time() - 1850.0)),
        normalize="none",
        check_reciprocal=False,
    ),
)
mix_model = FlowModel(pmap2)
mix_model.add_flow(TransitionFlow("inf", state2["S"], state2["I"], foi))
compiled_mix = mix_model.compile()
y2 = np.array([900.0, 800.0, 80.0, 10.0, 0.0, 0.0])
# 1851.2 is year index 1, whose rows are identical, so both ages share one FOI.
foi_mid = np.asarray(
    compiled_mix.observe(1851.2, y2, {"stack": stack}).captures["infection"].data
)
np.testing.assert_allclose(foi_mid[0], foi_mid[1], rtol=1e-5, atol=1e-6)
# 1840 clamps to the first, assortative, matrix.
foi_early = np.asarray(
    compiled_mix.observe(1840.0, y2, {"stack": stack}).captures["infection"].data
)
assert float(np.max(np.abs(foi_early[0] - foi_early[1]))) > 0.0
print(f"FOI in 1851 {foi_mid}, FOI in 1840 {foi_early}")

pd.DataFrame(
    {
        "1840 (clamped, assortative)": np.asarray(foi_early).reshape(-1),
        "1851 (homogeneous year)": np.asarray(foi_mid).reshape(-1),
    },
    index=list(age2.traits),
).plot.bar(
    title="Lookup gathers the mixing matrix for floor(year - 1850)",
    labels={"index": "age", "value": "force of infection"},
)


## Ageing from breakpoints

Both TB ports age people through uneven bands at rate one over the band width.
Trait names are the numeric lower bounds (`"0"`, `"5"`, `"15"`).
`TraitChain.from_breakpoints` builds the same chain as writing the pairs and
`1/width` rates by hand: the compiled digests match. Over forty years the
youngest band should empty into the older ones; the open-ended top band has no
outgoing edge.


In [ ]:
from summer4 import Compartments, SavePlan, SaveRequest, TraitChain

age3 = Property("age", ("0", "5", "15"))
state3 = Property("state", ("S",))
pmap3 = PropertyMap.from_property(state3).stratify(age3)
hand = TraitChain(age3, (("0", "5"), ("5", "15")), rates=(1.0 / 5.0, 1.0 / 10.0))
sugar = TraitChain.from_breakpoints(age3)
assert sugar == hand


def _ageing_digest(pairing: TraitChain) -> bytes:
    model = FlowModel(pmap3)
    model.add_flow(
        TransitionFlow("ageing", age3.present(), age3.present(), 1.0, pairing=pairing)
    )
    return model.compile()._digest


assert _ageing_digest(sugar) == _ageing_digest(hand)

age_model = FlowModel(pmap3)
age_model.add_flow(
    TransitionFlow("ageing", age3.present(), age3.present(), 1.0, pairing=sugar)
)
age_model.set_initial_population({age3["0"]: 1000.0, age3["5"]: 0.0, age3["15"]: 0.0})
age_compiled = age_model.compile()
times_age = np.linspace(0.0, 40.0, 81)
plan_age = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times_age)
age_res = age_compiled.run({}, t0=0.0, t1=40.0, dt=0.5, save=plan_age, solver="euler")
by_age = pd.DataFrame(
    {
        band: np.asarray(age_res["comp"].select(age3[band]).values.data).reshape(-1)
        for band in age3.traits
    },
    index=times_age,
)
assert float(by_age["0"].iloc[-1]) < float(by_age["0"].iloc[0])
assert float(by_age["15"].iloc[-1]) > float(by_age["15"].iloc[0])
print(
    f"at t=40: age 0 → {by_age['0'].iloc[-1]:.1f}, "
    f"5 → {by_age['5'].iloc[-1]:.1f}, 15 → {by_age['15'].iloc[-1]:.1f}"
)

by_age.plot(
    title="Ageing at 1/width: people leave 0→5→15; top band has no exit",
    labels={"index": "time (years)", "value": "people"},
)
